# Day 21 — Error Handling Mastery
 
**Week 4: Object-Oriented Programming & Modules**
 
Topics covered: Python exception hierarchy, built-in exception types, `try`/`except`/`else`/`finally`, exception groups (Python 3.11+), context managers for resource cleanup, `contextlib`, custom exception hierarchies with OOP, exception chaining, logging errors, defensive programming patterns.
 
---

### Q1. Mapping Built-in Exceptions to Scenarios
Without writing runnable code first, write a comment block listing which built-in exception each of these situations raises: indexing a list out of range, dividing by zero, calling a method on `None`, opening a missing file, converting `"abc"` to `int`, accessing a missing dictionary key, importing a missing module. Then verify each one with a small `try/except` block.
 

In [1]:
# Q1 — Exception Mapping
# ─────────────────────────────────────────────
# list[99]          → IndexError
# 1 / 0             → ZeroDivisionError
# None.method()     → AttributeError
# open("missing")   → FileNotFoundError  (subclass of OSError)
# int("abc")        → ValueError
# {}["key"]         → KeyError
# import missing    → ModuleNotFoundError (subclass of ImportError)

scenarios = [
    ("IndexError",          lambda: [][99]),
    ("ZeroDivisionError",   lambda: 1 / 0),
    ("AttributeError",      lambda: None.strip()),
    ("FileNotFoundError",   lambda: open("__no_such_file__")),
    ("ValueError",          lambda: int("abc")),
    ("KeyError",            lambda: {}["key"]),
    ("ModuleNotFoundError", lambda: __import__("__no_such_module__")),
]

for name, trigger in scenarios:
    try:
        trigger()
    except Exception as e:
        match = "✓" if type(e).__name__ == name else "✗"
        print(f"{match} {name:25s} → {type(e).__name__}: {e}")

✓ IndexError                → IndexError: list index out of range
✓ ZeroDivisionError         → ZeroDivisionError: division by zero
✓ AttributeError            → AttributeError: 'NoneType' object has no attribute 'strip'
✓ FileNotFoundError         → FileNotFoundError: [Errno 2] No such file or directory: '__no_such_file__'
✓ ValueError                → ValueError: invalid literal for int() with base 10: 'abc'
✓ KeyError                  → KeyError: 'key'
✓ ModuleNotFoundError       → ModuleNotFoundError: No module named '__no_such_module__'


- FileNotFoundError → OSError → Exception — catch OSError to handle all I/O errors
- ModuleNotFoundError → ImportError — except ImportError catches both
- All seven are direct subclasses of Exception, never catch bare BaseException unless you mean it

### Q2. Exception Hierarchy Tree
Using Python, programmatically walk and print the exception class hierarchy starting from `BaseException`, indented by level. (Hint: write a recursive function that uses `__subclasses__()`.)

In [2]:
def print_hierarchy(cls, indent=0):
    print(" " * indent + cls.__name__)
    for sub in cls.__subclasses__():
        print_hierarchy(sub, indent + 4)

print_hierarchy(BaseException)

BaseException
    BaseExceptionGroup
        ExceptionGroup
    Exception
        ArithmeticError
            FloatingPointError
            OverflowError
            ZeroDivisionError
                DivisionByZero
                DivisionUndefined
            DecimalException
                Clamped
                Rounded
                    Underflow
                    Overflow
                Inexact
                    Underflow
                    Overflow
                Subnormal
                    Underflow
                DivisionByZero
                FloatOperation
                InvalidOperation
                    ConversionSyntax
                    DivisionImpossible
                    DivisionUndefined
                    InvalidContext
        AssertionError
        AttributeError
            FrozenInstanceError
        BufferError
        EOFError
            IncompleteReadError
        ImportError
            ModuleNotFoundError
                PackageNotFoundE

- BaseException has 4 direct children — only Exception is meant for normal error handling
- KeyboardInterrupt / SystemExit / GeneratorExit sit outside Exception — except Exception won't swallow them
- OSError unifies all I/O errors; FileNotFoundError, PermissionError, etc. are just named subclasses
- LookupError is the common parent of IndexError + KeyError — useful for catching both at once

### Q3. Catching the Right Exception
Write a function `process(data)` that does three things that could each fail differently (index access, integer conversion, dictionary key access). Catch each possible exception type separately and print which operation failed without masking the others.

In [3]:
def process(data):
    # op 1: index access
    try:
        first = data["values"][0]
    except KeyError as e:
        print(f"Missing key: {e}")
        return
    except IndexError:
        print("'values' list is empty")
        return

    # op 2: int conversion
    try:
        number = int(first)
    except ValueError:
        print(f"Cannot convert {first!r} to int")
        return

    # op 3: dict key access
    try:
        label = data["labels"][number]
    except KeyError:
        print(f"No 'labels' key in data")
        return
    except IndexError:
        print(f"Index {number} out of range for 'labels'")
        return

    print(f"Result: {label}")

# usage
process({"values": ["2"], "labels": ["a", "b", "c"]})  # Result: c
process({"values": [],    "labels": ["a"]})              # 'values' list is empty
process({"values": ["x"], "labels": ["a"]})              # Cannot convert 'x' to int
process({"values": ["9"], "labels": ["a"]})              # Index 9 out of range for 'labels'
process({})                                              # Missing key: 'values'

Result: c
'values' list is empty
Cannot convert 'x' to int
Index 9 out of range for 'labels'
Missing key: 'values'


- Each try block wraps exactly one operation — wide try blocks hide which line failed
- Early return after each failure keeps nesting flat; no elif chains needed
- IndexError and KeyError share parent LookupError — catch separately here since the messages differ

### Q4. `else` Clause Deep Dive
Write a function where the `else` block of a `try/except` is non-trivially useful — a case where you do real work in `else` that should only happen if no exception occurred, and which itself could raise a different exception that should NOT be caught by the `except` above it. Demonstrate that distinction clearly.

In [7]:
import json

def load_and_transform(filepath, key):
    try:
        with open(filepath) as f:
            data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"[load failed] {type(e).__name__}: {e}")
        return None
    else:
        value = data[key]
        result = int(value) * 100
        print(f"[ok] {key} = {result}")
        return result

import tempfile, os

with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as f:
    json.dump({"score": "7"}, f)
    good = f.name

with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as f:
    f.write("not json{{{")
    bad = f.name

print("1:", load_and_transform(good, "score"))
print("2:", load_and_transform("/no/file.json", "x"))
print("3:", load_and_transform(bad, "x"))

try:
    load_and_transform(good, "missing_key")
except KeyError as e:
    print(f"[bubbled] KeyError {e} escaped the except block ✓")

with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as f:
    json.dump({"score": "abc"}, f)
    corrupt = f.name

try:
    load_and_transform(corrupt, "score")
except ValueError as e:
    print(f"[bubbled] ValueError '{e}' escaped the except block ✓")

os.unlink(good); os.unlink(bad); os.unlink(corrupt)

[ok] score = 700
1: 700
[load failed] FileNotFoundError: [Errno 2] No such file or directory: '/no/file.json'
2: None
[load failed] JSONDecodeError: Expecting value: line 1 column 1 (char 0)
3: None
[bubbled] KeyError 'missing_key' escaped the except block ✓
[bubbled] ValueError 'invalid literal for int() with base 10: 'abc'' escaped the except block ✓


- except guards only I/O and parse failures — errors that mean "we have nothing to work with"
- else runs the transform — errors there mean "we have data but it's wrong" — a different category
- Putting transform inside try would silently swallow KeyError/ValueError as if the file was bad — wrong semantics
- Rule: else = "do this only on success, but don't protect it from its own errors"

### Q5. `finally` for Resource Cleanup
Simulate a scenario with a "database connection" (just a class with `open()` and `close()` methods) where `finally` guarantees `close()` is always called even if an exception occurs during processing. Demonstrate with a successful and a failing execution.

In [8]:
class DBConnection:
    def __init__(self, dsn):
        self.dsn = dsn
        self.open = False

    def connect(self):
        print(f"[db] connected to {self.dsn}")
        self.open = True

    def query(self, sql):
        if not self.open:
            raise RuntimeError("Query on closed connection")
        if "DROP" in sql.upper():
            raise PermissionError(f"Forbidden: {sql!r}")
        print(f"[db] query ok: {sql!r}")
        return ["row1", "row2"]

    def close(self):
        self.open = False
        print(f"[db] connection closed")


def run_query(dsn, sql):
    db = DBConnection(dsn)
    try:
        db.connect()
        result = db.query(sql)
        print(f"[app] got {len(result)} rows")
    except PermissionError as e:
        print(f"[app] caught: {e}")
    finally:
        db.close()  # always runs


print("=== success ===")
run_query("prod:5432/app", "SELECT * FROM users")

print("\n=== failure ===")
run_query("prod:5432/app", "DROP TABLE users")

=== success ===
[db] connected to prod:5432/app
[db] query ok: 'SELECT * FROM users'
[app] got 2 rows
[db] connection closed

=== failure ===
[db] connected to prod:5432/app
[app] caught: Forbidden: 'DROP TABLE users'
[db] connection closed


- finally runs whether the try succeeds, an except catches something, or an uncaught exception propagates up — connection always closes
- except here is optional; even without it finally still fires before the exception unwinds the stack
- This pattern is exactly what with/__exit__ automates — Q6 will show that cleaner form

### Q6. Exception Chaining — `raise X from Y`
Write a function `load_config(path)` that catches `FileNotFoundError` and re-raises a custom `ConfigLoadError` using `raise ConfigLoadError(...) from original_exc`. Print the traceback to show the chain. Then suppress chaining with `raise X from None` and compare the output.

In [9]:
import traceback

class ConfigLoadError(Exception):
    pass


def load_config(path, suppress_chain=False):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError as e:
        if suppress_chain:
            raise ConfigLoadError(f"Config not found: {path!r}") from None
        else:
            raise ConfigLoadError(f"Config not found: {path!r}") from e


print("=== with chaining (raise X from Y) ===")
try:
    load_config("missing.cfg")
except ConfigLoadError:
    traceback.print_exc()

print("\n=== suppressed chaining (raise X from None) ===")
try:
    load_config("missing.cfg", suppress_chain=True)
except ConfigLoadError:
    traceback.print_exc()

=== with chaining (raise X from Y) ===

=== suppressed chaining (raise X from None) ===


Traceback (most recent call last):
  File "/tmp/ipykernel_44738/2007268822.py", line 9, in load_config
    with open(path) as f:
         ^^^^^^^^^^
  File "/workspaces/Python-to-ML-Roadmap/venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 346, in _modified_open
    return io_open(file, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'missing.cfg'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/tmp/ipykernel_44738/2007268822.py", line 20, in <module>
    load_config("missing.cfg")
  File "/tmp/ipykernel_44738/2007268822.py", line 15, in load_config
    raise ConfigLoadError(f"Config not found: {path!r}") from e
ConfigLoadError: Config not found: 'missing.cfg'
Traceback (most recent call last):
  File "/tmp/ipykernel_44738/2007268822.py", line 26, in <module>
    load_config("missing.cfg", suppress_chain=True)
  File "/tmp/ipykerne

- from e — API boundary: wrapping low-level error into domain error, caller needs the root cause
- bare raise X — implicit chain, Python sets __context__ automatically; fine for internal re-raises
- from None — when the original exception is noise/security leak (e.g. hiding DB internals from HTTP responses)

### Q7. Suppressing Exceptions with `contextlib.suppress`
Use `contextlib.suppress(FileNotFoundError)` to silently ignore a missing file deletion, rather than wrapping it in a `try/except`. Show how this is cleaner for "I don't care if this fails" cases.

In [10]:
import os
from contextlib import suppress

# verbose try/except — 4 lines to say "I don't care"
try:
    os.remove("ghost.tmp")
except FileNotFoundError:
    pass

# suppress — 2 lines, intent is explicit
with suppress(FileNotFoundError):
    os.remove("ghost.tmp")


# realistic use: cleanup in a temp-file workflow
def process_file(path):
    tmp = path + ".lock"
    try:
        open(tmp, "w").close()
        print(f"[work] processing {path!r}")
        if not os.path.exists(path):
            raise ValueError(f"source missing: {path!r}")
        print("[work] done")
    finally:
        with suppress(FileNotFoundError):  # lock may already be gone — don't care
            os.remove(tmp)
        print(f"[cleanup] lock released (if it existed)")


process_file("data.csv")        # source missing — ValueError still propagates

[work] processing 'data.csv'
[cleanup] lock released (if it existed)


ValueError: source missing: 'data.csv'

### Q8. `warnings` Module — Non-Fatal Alerts
Use the `warnings` module to issue a `DeprecationWarning` inside a function that is being replaced by a newer version. Show how to: issue the warning, filter it to turn it into an error, and suppress it.

### suppress only swallows what you list — ValueError still propagates. Rule of thumb:

| <b> Reach for suppress when… </b> | <b> Stick with try/except when… </b> |
|---|---|
| failure = expected no-op (rm, mkdir, unlink) | you need to log, recover, or return a fallback | 
| one-liner body, zero recovery logic | multiple exception types with different handlers | 
| intent is "best effort" | you need the exception object (as e)|

### Q9. Custom Exception Hierarchy (3-Level Deep)
Design a 3-level exception hierarchy for a "payment processing" system:
- `PaymentError(Exception)` — base
  - `CardError(PaymentError)` — card-related issues
    - `CardDeclinedError(CardError)`
    - `CardExpiredError(CardError)`
  - `InsufficientFundsError(PaymentError)`
Write functions that raise each leaf exception. Write a handler that catches `CardError` in one block and `PaymentError` (for anything else) in another, showing the hierarchy allows grouping.
 

In [11]:
class PaymentError(Exception):
    def __init__(self, msg, amount=None):
        super().__init__(msg)
        self.amount = amount

class CardError(PaymentError):
    def __init__(self, msg, last4=None, **kw):
        super().__init__(msg, **kw)
        self.last4 = last4

class CardDeclinedError(CardError):
    pass

class CardExpiredError(CardError):
    def __init__(self, msg, expiry=None, **kw):
        super().__init__(msg, **kw)
        self.expiry = expiry

class InsufficientFundsError(PaymentError):
    def __init__(self, msg, shortfall=None, **kw):
        super().__init__(msg, **kw)
        self.shortfall = shortfall


# --- raisers ---
def charge(card_no, amount, scenario):
    match scenario:
        case "declined":
            raise CardDeclinedError("Card declined by issuer", last4=card_no[-4:], amount=amount)
        case "expired":
            raise CardExpiredError("Card expired", last4=card_no[-4:], expiry="11/23", amount=amount)
        case "funds":
            raise InsufficientFundsError("Insufficient funds", amount=amount, shortfall=amount - 10.00)
        case _:
            print(f"[ok] charged ${amount:.2f}")


# --- handler showing hierarchy grouping ---
def process_payment(card_no, amount, scenario):
    try:
        charge(card_no, amount, scenario)
    except CardError as e:
        # catches CardDeclinedError + CardExpiredError — card-specific UX
        print(f"[card error] {e} (card *{e.last4})")
        if isinstance(e, CardExpiredError):
            print(f"  → expired: {e.expiry}, ask customer to update card")
        else:
            print(f"  → advise customer to contact their bank")
    except PaymentError as e:
        # catches InsufficientFundsError — anything else payment-related
        print(f"[payment error] {e}")
        if isinstance(e, InsufficientFundsError):
            print(f"  → shortfall: ${e.shortfall:.2f}, suggest partial payment of ${e.amount - e.shortfall:.2f}")


card = "4111111111111234"
for scenario in ("ok", "declined", "expired", "funds"):
    print(f"--- {scenario} ---")
    process_payment(card, 99.99, scenario)

--- ok ---
[ok] charged $99.99
--- declined ---
[card error] Card declined by issuer (card *1234)
  → advise customer to contact their bank
--- expired ---
[card error] Card expired (card *1234)
  → expired: 11/23, ask customer to update card
--- funds ---
[payment error] Insufficient funds
  → shortfall: $89.99, suggest partial payment of $10.00


- Always catch specific before general — `CardError` before `PaymentError`, or the broad handler swallows everything
- `except CardError` catches all three card types; `except PaymentError` is the fallback net
- `isinstance()` inside a handler lets you branch without adding more `except` blocks
- Base classes carry shared fields `(amount)`; subclasses add their own `(last4, expiry, shortfall)`

### Q10. Storing Extra Data on Exceptions
Create a custom exception `APIError` that stores `status_code` (int) and `endpoint` (str) as attributes set in `__init__`. Override `__str__` to produce a formatted message. Raise and catch it, accessing both attributes in the handler.

In [12]:
class APIError(Exception):
    def __init__(self, status_code, endpoint, detail=""):
        self.status_code = status_code
        self.endpoint = endpoint
        self.detail = detail
        super().__init__(str(self))  # makes str(e) and repr(e) consistent

    def __str__(self):
        base = f"HTTP {self.status_code} @ {self.endpoint!r}"
        return f"{base} — {self.detail}" if self.detail else base


def fetch(endpoint, simulate_status):
    match simulate_status:
        case 401: raise APIError(401, endpoint, "missing or invalid token")
        case 404: raise APIError(404, endpoint, "resource not found")
        case 429: raise APIError(429, endpoint, "rate limit exceeded")
        case 500: raise APIError(500, endpoint, "internal server error")
        case 200: print(f"[ok] 200 @ {endpoint!r}")


for status in (200, 401, 404, 429, 500):
    try:
        fetch("/api/users", status)
    except APIError as e:
        print(f"[caught] {e}")
        print(f"         status={e.status_code}  endpoint={e.endpoint!r}  detail={e.detail!r}")
        if e.status_code == 429:
            print(f"         → back off and retry")
        elif e.status_code >= 500:
            print(f"         → page on-call")

[ok] 200 @ '/api/users'
[caught] HTTP 401 @ '/api/users' — missing or invalid token
         status=401  endpoint='/api/users'  detail='missing or invalid token'
[caught] HTTP 404 @ '/api/users' — resource not found
         status=404  endpoint='/api/users'  detail='resource not found'
[caught] HTTP 429 @ '/api/users' — rate limit exceeded
         status=429  endpoint='/api/users'  detail='rate limit exceeded'
         → back off and retry
[caught] HTTP 500 @ '/api/users' — internal server error
         status=500  endpoint='/api/users'  detail='internal server error'
         → page on-call


- `super().__init__(str(self))` — passes formatted string to `Exception.args[0]`, so `str(e), repr(e)`, and `logging.exception()` all show the full message without extra work
- attributes on the exception object let handlers make programmatic decisions `(e.status_code >= 500)` without parsing strings
- `detail=""` default keeps `APIError(503, "/health")` valid without boilerplate

### Q11. Re-Raising Inside a Broad `except`
Write a function that catches all exceptions with `except Exception as e`, logs the error type and message to a list (simulating a log), and then re-raises the original exception. Show the caller still receives the exception normally.

In [13]:
log = []

def logged(func, *args, **kwargs):
    try:
        return func(*args, **kwargs)
    except Exception as e:
        log.append({"type": type(e).__name__, "msg": str(e)})
        raise  # bare raise — preserves original traceback


def divide(a, b): return a / b
def parse(s): return int(s)
def fetch(key): return {}[key]

for fn, args in [(divide, (10, 0)), (parse, ("x",)), (fetch, ("missing",))]:
    try:
        logged(fn, *args)
    except Exception as e:
        print(f"[caller] caught {type(e).__name__}: {e}")

print("\n[log]")
for entry in log:
    print(f"  {entry}")

[caller] caught ZeroDivisionError: division by zero
[caller] caught ValueError: invalid literal for int() with base 10: 'x'
[caller] caught KeyError: 'missing'

[log]
  {'type': 'ZeroDivisionError', 'msg': 'division by zero'}
  {'type': 'ValueError', 'msg': "invalid literal for int() with base 10: 'x'"}
  {'type': 'KeyError', 'msg': "'missing'"}


### `raise` vs `raise e` — critical difference:
Traceback points to 
|   |   |
|---|---|
| raise | original line that raised — correct ✓ |
| raise e | the raise e line inside except — misleading ✗ |

Bare `raise` with no argument is the only correct way to re-raise inside an `except` block — it replays the original exception object, type, and traceback unmodified.

## Q12. Writing a Context Manager Class
Write a `DatabaseTransaction` class (simulated) implementing `__enter__` and `__exit__`. `__enter__` returns a connection-like object; `__exit__` commits if no exception occurred, or rolls back and suppresses the exception if a specific `RollbackError` was raised (returns `True`), but lets all other exceptions propagate (returns `False`).

In [18]:
class RollbackError(Exception):
    pass


class FakeConnection:
    def execute(self, sql):
        print(f"  [sql] {sql}")


class DatabaseTransaction:
    def __init__(self, dsn):
        self.dsn = dsn
        self.conn = None

    def __enter__(self):
        print(f"[tx] open — {self.dsn}")
        self.conn = FakeConnection()
        return self.conn

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is None:
            print("[tx] commit")
            return False                    # no exception, nothing to suppress

        if exc_type is RollbackError:
            print(f"[tx] rollback — {exc_val}")
            return True                     # suppresses RollbackError

        print(f"[tx] rollback — unexpected {exc_type.__name__}, re-raising")
        return False                        # propagates everything else


# --- success ---
print("=== success ===")
with DatabaseTransaction("prod:5432") as conn:
    conn.execute("INSERT INTO orders VALUES (1, 99.99)")
    conn.execute("UPDATE inventory SET stock = stock - 1")

# --- RollbackError: suppressed ---
print("\n=== RollbackError (suppressed) ===")
with DatabaseTransaction("prod:5432") as conn:
    conn.execute("INSERT INTO orders VALUES (2, 49.99)")
    raise RollbackError("stock reservation failed")
print("[caller] continues normally after suppressed RollbackError ✓")

# --- other exception: propagates ---
print("\n=== ValueError (propagates) ===")
try:
    with DatabaseTransaction("prod:5432") as conn:
        conn.execute("INSERT INTO orders VALUES (3, -1.00)")
        raise ValueError("negative amount not allowed")
except ValueError as e:
    print(f"[caller] caught {type(e).__name__}: {e} ✓")

=== success ===
[tx] open — prod:5432
  [sql] INSERT INTO orders VALUES (1, 99.99)
  [sql] UPDATE inventory SET stock = stock - 1
[tx] commit

=== RollbackError (suppressed) ===
[tx] open — prod:5432
  [sql] INSERT INTO orders VALUES (2, 49.99)
[tx] rollback — stock reservation failed
[caller] continues normally after suppressed RollbackError ✓

=== ValueError (propagates) ===
[tx] open — prod:5432
  [sql] INSERT INTO orders VALUES (3, -1.00)
[tx] rollback — unexpected ValueError, re-raising
[caller] caught ValueError: negative amount not allowed ✓


### `__exit__` signature and return value rules:
|   |   |   |
|---|---|---|
| `exc_type` | return `True` | return `False` | 
| `None` (no exception) | no effect | no effect |
| matched exception | suppressed — execution continues after `with` | propagates to caller
| unmatched exception | suppressed (dangerous — swallows bugs) | propagates ✓ |

- `__exit__` always receives `(exc_type, exc_val, exc_tb)` — all three are `None` on clean exit
- `return True` is a deliberate swallow — only do it for exceptions you fully own and understand
- rollback fires on both `RollbackError` and `ValueError` — cleanup is unconditional, suppression is selective

### Q14. `contextlib.contextmanager` — Function-Based Context Manager
Rewrite the `DatabaseTransaction` from Q13 as a generator-based context manager using `@contextlib.contextmanager`, using `try/yield/except/finally` inside the generator.

In [19]:
from contextlib import contextmanager

class RollbackError(Exception):
    pass

class FakeConnection:
    def execute(self, sql):
        print(f"  [sql] {sql}")

@contextmanager
def db_transaction(dsn):
    conn = FakeConnection()
    print(f"[tx] open — {dsn}")
    try:
        yield conn                          # __enter__ returns here
        print("[tx] commit")               # runs only if no exception
    except RollbackError as e:
        print(f"[tx] rollback — {e}")
                                           # swallowed: generator returns normally
    except Exception as e:
        print(f"[tx] rollback — unexpected {type(e).__name__}, re-raising")
        raise                              # __exit__ returns False equivalent
    finally:
        print("[tx] connection closed")    # always runs


# --- success ---
print("=== success ===")
with db_transaction("prod:5432") as conn:
    conn.execute("INSERT INTO orders VALUES (1, 99.99)")

# --- RollbackError: suppressed ---
print("\n=== RollbackError (suppressed) ===")
with db_transaction("prod:5432") as conn:
    conn.execute("INSERT INTO orders VALUES (2, 49.99)")
    raise RollbackError("stock reservation failed")
print("[caller] continues normally ✓")

# --- other exception: propagates ---
print("\n=== ValueError (propagates) ===")
try:
    with db_transaction("prod:5432") as conn:
        conn.execute("INSERT INTO orders VALUES (3, -1.00)")
        raise ValueError("negative amount")
except ValueError as e:
    print(f"[caller] caught {type(e).__name__}: {e} ✓")

=== success ===
[tx] open — prod:5432
  [sql] INSERT INTO orders VALUES (1, 99.99)
[tx] commit
[tx] connection closed

=== RollbackError (suppressed) ===
[tx] open — prod:5432
  [sql] INSERT INTO orders VALUES (2, 49.99)
[tx] rollback — stock reservation failed
[tx] connection closed
[caller] continues normally ✓

=== ValueError (propagates) ===
[tx] open — prod:5432
  [sql] INSERT INTO orders VALUES (3, -1.00)
[tx] rollback — unexpected ValueError, re-raising
[tx] connection closed
[caller] caught ValueError: negative amount ✓


### Class vs generator — direct mapping:
|   |   |
|---|---|
| `__dunder__` class | `@contextmanager` generator 
| `__enter__` | everything before `yield` |
| `yield value` | return value of `as` target |
| `__exit__` success path | code after `yield` (no exception) |
| `__exit__` exc handling | `except` blocks around `yield` |
| `return False` (propagate) | `raise` inside `except` |
| `return True` (suppress) | catch and don't re-raise |
| teardown | finally block |

### When to pick each:

- `@contextmanager` — less boilerplate, linear read top-to-bottom, ideal for simple acquire/release
- `__enter__/__exit__` class — subclassable, storable as object, better when the manager carries meaningful state across calls

### Q15. `contextlib.ExitStack`
Use `contextlib.ExitStack` to open a dynamic number of files (determined at runtime from a list of filenames) within a single `with` block, ensuring all are closed properly even if processing fails partway through.

In [20]:
import os
import tempfile
from contextlib import ExitStack

# setup: create temp files with content
paths = []
for i, content in enumerate(["alpha\n", "beta\n", "gamma\n"]):
    f = tempfile.NamedTemporaryFile("w", suffix=f"_{i}.txt", delete=False)
    f.write(content)
    f.close()
    paths.append(f.name)


def process_files(filenames, fail_at=None):
    print(f"opening {len(filenames)} files...")
    with ExitStack() as stack:
        handles = [stack.enter_context(open(p)) for p in filenames]
        print(f"all open: {[h.name.split('/')[-1] for h in handles]}")
        for i, fh in enumerate(handles):
            if i == fail_at:
                raise RuntimeError(f"processing failed at file {i}")
            print(f"  read: {fh.read().strip()!r}")
    print("all closed ✓\n")


# success: all files processed and closed
print("=== success ===")
process_files(paths)

# failure midway: all files still closed
print("=== failure at file 1 ===")
try:
    process_files(paths, fail_at=1)
except RuntimeError as e:
    print(f"[caller] caught: {e}")
    print("all closed despite failure ✓\n")

# dynamic subset: runtime-determined file list
print("=== dynamic subset (last 2 only) ===")
process_files(paths[-2:])

# cleanup
for p in paths:
    os.unlink(p)

=== success ===
opening 3 files...
all open: ['tmpbaxhv3oz_0.txt', 'tmp98cqwztv_1.txt', 'tmp70pl_gv6_2.txt']
  read: 'alpha'
  read: 'beta'
  read: 'gamma'
all closed ✓

=== failure at file 1 ===
opening 3 files...
all open: ['tmpbaxhv3oz_0.txt', 'tmp98cqwztv_1.txt', 'tmp70pl_gv6_2.txt']
  read: 'alpha'
[caller] caught: processing failed at file 1
all closed despite failure ✓

=== dynamic subset (last 2 only) ===
opening 2 files...
all open: ['tmp98cqwztv_1.txt', 'tmp70pl_gv6_2.txt']
  read: 'beta'
  read: 'gamma'
all closed ✓



### Why `ExitStack` — the alternatives fail for dynamic counts:
|   |   |
|---|---|
| approach | problem |
| nested with `open(a), open(b)` |count fixed at write-time |
| manual `f.close()` in `finally` | skipped if open itself raises midway | 
| `ExitStack` | registers each handle as opened; closes all in LIFO order on exit regardless of where failure occurred |
---

- `stack.enter_context(cm)` is equivalent to `__enter__` — registers the cleanup and returns the value
- closes in reverse registration order (LIFO) — mirrors nested `with` semantics
- also accepts callbacks via `stack.callback(fn)` for non-context-manager resources

### Q16. Logging Errors with the `logging` Module
Set up a `logging` configuration that writes ERROR and above to a file and INFO and above to the console. Write a function that catches exceptions and logs them using `logging.exception()` (which automatically includes the traceback). Show the difference between `logging.error()` and `logging.exception()`.
 

In [24]:
import logging

# --- setup ---
logger = logging.getLogger("app")
logger.setLevel(logging.DEBUG)

fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")

console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(fmt)

file_handler = logging.FileHandler("app.log", mode="w")
file_handler.setLevel(logging.ERROR)
file_handler.setFormatter(fmt)

logger.addHandler(console)
logger.addHandler(file_handler)


# --- difference: error() vs exception() ---
def divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        logger.error("division failed — no traceback")          # message only
        logger.exception("division failed — with traceback")    # message + exc_info
        return None


def parse_config(data):
    try:
        return {"timeout": int(data["timeout"]), "retries": int(data["retries"])}
    except KeyError as e:
        logger.exception("missing config key")
        return None
    except ValueError as e:
        logger.exception("invalid config value")
        return None


print("=== console output ===")
divide(10, 0)
print()
parse_config({"timeout": "5", "retries": "abc"})
print()
parse_config({"timeout": "5"})

# show file captured ERROR+ only
print("\n=== app.log contents ===")
with open("app.log") as f:
    print(f.read())

01:12:39 [ERROR] division failed — no traceback
01:12:39 [ERROR] division failed — no traceback
01:12:39 [ERROR] division failed — no traceback
01:12:39 [ERROR] division failed — no traceback
01:12:39 [ERROR] division failed — with traceback
Traceback (most recent call last):
  File "/tmp/ipykernel_44738/3270470961.py", line 24, in divide
    return a / b
           ~~^~~
ZeroDivisionError: division by zero
01:12:39 [ERROR] division failed — with traceback
Traceback (most recent call last):
  File "/tmp/ipykernel_44738/3270470961.py", line 24, in divide
    return a / b
           ~~^~~
ZeroDivisionError: division by zero
01:12:39 [ERROR] division failed — with traceback
Traceback (most recent call last):
  File "/tmp/ipykernel_44738/3270470961.py", line 24, in divide
    return a / b
           ~~^~~
ZeroDivisionError: division by zero
01:12:39 [ERROR] division failed — with traceback
Traceback (most recent call last):
  File "/tmp/ipykernel_44738/3270470961.py", line 24, in divide
  

=== console output ===



=== app.log contents ===
01:12:39 [ERROR] division failed — no traceback
01:12:39 [ERROR] division failed — with traceback
Traceback (most recent call last):
  File "/tmp/ipykernel_44738/3270470961.py", line 24, in divide
    return a / b
           ~~^~~
ZeroDivisionError: division by zero
01:12:39 [ERROR] invalid config value
Traceback (most recent call last):
  File "/tmp/ipykernel_44738/3270470961.py", line 33, in parse_config
    return {"timeout": int(data["timeout"]), "retries": int(data["retries"])}
                                                        ^^^^^^^^^^^^^^^^^^^^
ValueError: invalid literal for int() with base 10: 'abc'
01:12:39 [ERROR] missing config key
Traceback (most recent call last):
  File "/tmp/ipykernel_44738/3270470961.py", line 33, in parse_config
    return {"timeout": int(data["timeout"]), "retries": int(data["retries"])}
                                                            ~~~~^^^^^^^^^^^
KeyError: 'retries'
/ipykernel_

In [25]:
import logging

logger = logging.getLogger("app")
logger.handlers.clear()          # ← remove handlers from previous runs
logger.setLevel(logging.DEBUG)

fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")

console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(fmt)

file_handler = logging.FileHandler("app.log", mode="w")
file_handler.setLevel(logging.ERROR)
file_handler.setFormatter(fmt)

logger.addHandler(console)
logger.addHandler(file_handler)
logger.propagate = False         # ← stop bubbling to root logger (another dupe source)

In [26]:
logging.getLogger("app").handlers.clear()
logging.getLogger("app").propagate = False

### Q17. Global Exception Handler with `sys.excepthook`
Replace Python's default uncaught exception handler by assigning a custom function to `sys.excepthook`. The custom handler should log the exception to a file and print a user-friendly message instead of the raw traceback. Demonstrate with a deliberately unhandled exception. (In a notebook, note that `sys.excepthook` may behave differently — document this.)

### Q18. Exception Safety in Data Structures
Write a class `SafeList` wrapping a list. Its `append_all(items)` method attempts to append all items in a batch, but if any item fails a validation check (e.g., must be int), it rolls back ALL appends from that batch (atomicity) using exception handling, leaving the list unchanged. Demonstrate with a mixed-type input.

### Q19. Defensive Programming — EAFP vs LBYL
The Python community prefers EAFP ("Easier to Ask Forgiveness than Permission") over LBYL ("Look Before You Leap"). Rewrite the same data-access function twice: once in LBYL style (check types/existence before using) and once in EAFP style (just try it and handle exceptions). Provide a comparison comment on readability and edge cases.

### Q20. Exception Groups (Python 3.11+)
Use `ExceptionGroup` and `except*` syntax to handle multiple simultaneous exceptions from simulated parallel tasks. Create an `ExceptionGroup` with 3 different exception types, and demonstrate using `except* ValueError` and `except* TypeError` to handle each group separately. (If running Python < 3.11, add a comment explaining the syntax and why it was introduced.)
 

### Q21. Retry Logic with Exponential Backoff
Write a function `retry_with_backoff(func, max_retries=5, base_delay=0.1)` that calls `func()`, and if it raises an exception, waits `base_delay * (2 ** attempt)` seconds before retrying, up to `max_retries` times. Use `time.sleep`. Test it on a function that fails the first 3 times before succeeding (using a counter in a closure).

### Q22. Custom Exception with `__reduce__` for Pickling
Create a custom exception `SerializableError` that can be pickled and unpickled correctly (custom exceptions with extra `__init__` args often fail with `pickle` by default). Implement `__reduce__` to fix this. Demonstrate with `pickle.dumps` and `pickle.loads`.

### Q23. Exception-Safe Decorator
Write a decorator `exception_boundary` that catches any exception from the wrapped function, wraps it in a custom `BoundaryError` (with the original as cause), and raises that instead. This is the pattern used to create clean API boundaries — internal errors don't leak out as implementation details.

### Q24. Validation Chain with Accumulated Errors
Instead of failing on the first validation error, write a `validate_user(data)` function that runs all validations (name not empty, age is int and > 0, email contains `@`, password length >= 8) and collects ALL errors into a list, then raises a single `ValidationError` containing the full list of problems. This is the pattern used in frameworks like Django Forms and Pydantic.

### Q25. Challenge — Resilient Pipeline with Full Error Handling
Build a data processing pipeline that reads records from a JSON file, processes them through 3 validation stages, and writes valid results to an output CSV. The requirements:
1. Each stage has its own custom exception type
2. A record that fails stage 1 is skipped entirely; a record that fails stage 2 is partially processed and logged; a record that fails stage 3 is written to a "quarantine" file
3. All file operations use context managers
4. All errors are logged with the `logging` module (to both console and a file)
5. At the end, print a summary: total records, passed, failed per stage, quarantined
Create sample data with intentional failures at each stage to verify all paths work.
 